# 08b — Align velocyto loom layers and build the velocity-ready AnnData

Consumes notebook 08a exports plus WT/KO velocyto loom files and writes the exact H5AD used by notebooks 09 and 10. Source audited against `20250526p38-draw/sub/velocity.ipynb` (cell 2).


In [ ]:
# Centralized paths, deterministic seed, and scheduler-aware thread limits.
from pathlib import Path
import sys

repo_root = Path.cwd() if (Path.cwd() / "config" / "paths.py").exists() else Path.cwd().parent
if not (repo_root / "config" / "paths.py").exists():
    raise FileNotFoundError("Run Jupyter from the repository root or notebooks/ directory.")
sys.path.insert(0, str(repo_root))
from config.paths import P38_THREADS, RANDOM_SEED, p38_path, upstream_path


In [ ]:
import os
import warnings

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import io, sparse

np.random.seed(RANDOM_SEED)
warnings.filterwarnings("ignore", category=UserWarning, module="anndata._io.specs.loom")

export_dir = Path(p38_path("20250526p38-draw", "sub", "dynamo"))
wt_loom_path = Path(upstream_path("cellranger_sh_scripts_and_outputs", "WT", "WT_cellranger_output", "velocyto", "WT_cellranger_output.loom"))
ko_loom_path = Path(upstream_path("cellranger_sh_scripts_and_outputs", "KO", "KO_cellranger_output", "velocyto", "KO_cellranger_output.loom"))
velocity_h5ad = export_dir / "velocity_ready_full_with_detailed_index_check.h5ad"

required_files = [
    export_dir / "matrix_counts.mtx", export_dir / "matrix_normalized.mtx",
    export_dir / "genes.csv", export_dir / "barcodes.csv", export_dir / "metadata.csv",
    wt_loom_path, ko_loom_path,
]
missing_files = [str(path) for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError("Missing velocity inputs:\n" + "\n".join(missing_files))


In [ ]:
counts = io.mmread(export_dir / "matrix_counts.mtx").tocsr()
normalized = io.mmread(export_dir / "matrix_normalized.mtx").tocsr()
genes = pd.read_csv(export_dir / "genes.csv")["gene_name"].astype(str)
barcodes = pd.read_csv(export_dir / "barcodes.csv")["barcode"].astype(str)
metadata = pd.read_csv(export_dir / "metadata.csv", index_col=0)

if metadata.index.tolist() != barcodes.tolist():
    metadata = metadata.loc[barcodes].copy()
if counts.shape != normalized.shape or counts.shape != (len(genes), len(barcodes)):
    raise ValueError(f"Exported matrix/annotation dimensions disagree: {counts.shape}, {normalized.shape}")

adata = ad.AnnData(X=normalized.T.tocsr(), obs=metadata, var=pd.DataFrame(index=genes))
adata.obs_names = barcodes
adata.layers["counts"] = counts.T.tocsr()

for csv_name, key in (("pca.csv", "X_pca"), ("umap.csv", "X_umap")):
    csv_path = export_dir / csv_name
    if csv_path.exists():
        coordinates = pd.read_csv(csv_path, index_col=0).loc[adata.obs_names]
        adata.obsm[key] = coordinates.to_numpy()
pca_stdev_path = export_dir / "pca_stdev.csv"
if pca_stdev_path.exists():
    stdev = pd.read_csv(pca_stdev_path)["stdev"].to_numpy()
    adata.uns["pca"] = {"variance": stdev ** 2}

adata.var_names_make_unique()
print(f"Base AnnData: {adata.n_obs} cells x {adata.n_vars} genes")


In [ ]:
def clean_adata_barcode(barcode: str, sample: str) -> str:
    core = str(barcode)
    if core.startswith(sample + "_"):
        core = core[len(sample) + 1:]
    return core.split("-")[0]


def clean_loom_barcode(barcode: str) -> str:
    core = str(barcode).split(":")[-1]
    return core[:-1] if core.endswith("x") else core


def load_loom(path: Path) -> ad.AnnData:
    loom = sc.read_loom(path, sparse=True, cleanup=False, var_names="Gene")
    loom.var_names_make_unique()
    missing_layers = {"spliced", "unspliced"}.difference(loom.layers.keys())
    if missing_layers:
        raise KeyError(f"{path.name} lacks layers: {sorted(missing_layers)}")
    return loom


wt_loom = load_loom(wt_loom_path)
ko_loom = load_loom(ko_loom_path)


def aligned_layer_triplets(target: ad.AnnData, loom: ad.AnnData, sample: str, layer: str):
    target_rows = np.flatnonzero(target.obs["orig.ident"].astype(str).to_numpy() == sample)
    target_core = pd.Index([clean_adata_barcode(target.obs_names[i], sample) for i in target_rows])
    loom_core = pd.Index([clean_loom_barcode(x) for x in loom.obs_names])
    if target_core.has_duplicates or loom_core.has_duplicates:
        raise ValueError(f"Non-unique cleaned barcodes for {sample}")

    loom_lookup = pd.Series(np.arange(loom.n_obs), index=loom_core)
    matched = target_core.isin(loom_lookup.index)
    matched_target_rows = target_rows[matched]
    matched_loom_rows = loom_lookup.loc[target_core[matched]].to_numpy(dtype=int)

    common_genes = target.var_names.intersection(loom.var_names)
    target_cols = target.var_names.get_indexer(common_genes)
    loom_cols = loom.var_names.get_indexer(common_genes)
    if not len(matched_target_rows) or not len(common_genes):
        raise ValueError(f"No matched cells or genes for {sample}")

    block = sparse.coo_matrix(loom.layers[layer][matched_loom_rows, :][:, loom_cols])
    rows = matched_target_rows[block.row]
    cols = target_cols[block.col]
    print(f"{sample} {layer}: {len(matched_target_rows)}/{len(target_rows)} cells; {len(common_genes)} genes; {block.nnz} nonzero values")
    return rows, cols, block.data


for layer in ("spliced", "unspliced"):
    pieces = [aligned_layer_triplets(adata, wt_loom, "WT", layer), aligned_layer_triplets(adata, ko_loom, "KO", layer)]
    rows = np.concatenate([piece[0] for piece in pieces])
    cols = np.concatenate([piece[1] for piece in pieces])
    values = np.concatenate([piece[2] for piece in pieces])
    adata.layers[layer] = sparse.csr_matrix((values, (rows, cols)), shape=adata.shape)
    if adata.layers[layer].nnz == 0:
        raise ValueError(f"Constructed {layer} layer is empty")


In [ ]:
required_obs = {"orig.ident", "RNA_snn_res.0.8"}
missing_obs = required_obs.difference(adata.obs.columns)
if missing_obs:
    raise KeyError(f"Missing downstream metadata: {sorted(missing_obs)}")
if "X_umap" not in adata.obsm:
    raise KeyError("X_umap was not transferred from the Seurat export")

adata.uns["velocity_input_provenance"] = {
    "seurat_export": str(export_dir),
    "wt_loom": str(wt_loom_path),
    "ko_loom": str(ko_loom_path),
    "notebook": "08b_velocity_build_anndata.ipynb",
}
adata.write_h5ad(velocity_h5ad, compression="gzip")
print(f"Saved {velocity_h5ad} ({adata.n_obs} cells x {adata.n_vars} genes; layers={list(adata.layers)})")
